In [22]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-mpnet-base-v2")

# topic_text = """
# war, army, military, conflict, attack, bomb, invasion, battle, soldiers, violence,
# occupation, missile, fighting, defense, siege, casualties, combat, troops, conflict zones
# """
# topic_text = "government, parliament, prime minister, president, elections, policy, law, minister, democracy, legislation"
# topic_text = "economy, inflation, stock, market, business, trade, unemployment, budget, finance, investment, recession"
# topic_text = "protest, health, education, rights, inequality, social, law, welfare, healthcare, crime"
topic_text = "israel, palestine"
# topic_text = "russia, ukraine"

topic_emb = model.encode(topic_text, convert_to_tensor=True)

chunk = "By contrast, the bottom tier of technocrats appears relatively homogeneous – at least in terms of Palestinian identity – though it is reasonable to assume that both the Palestinian Authority and Hamas will exert pressure on each of its members to adopt positions closer to their own. The most difficult task is the demilitarization of Gaza and the disarmament of Hamas. At the Davos meeting, Jared Kushner presented a master plan for dismantling Hamas, demilitarizing the Strip, and rebuilding it, but it remains unclear who would carry out the first two tasks. The “international stabilization force” mentioned in Trump’s plan already has an American commander, but it has no army. Countries that committed to sending forces – such as Azerbaijan and Indonesia – have meanwhile backed out."
chunk_emb = model.encode(chunk, convert_to_tensor=True)

sim = util.cos_sim(chunk_emb, topic_emb)
print(sim)

tensor([[0.3524]])


In [ ]:
import pandas as pd
import re

class Months:
    MONTHS_WORDS1 = {"January": "01", "February": "02", "March": "03", "April": "04", "May": "05", "June": "06", "July": "07", "August": "08", "September": "09", "October": "10", "November": "11", "December": "12"}
    MONTHS_WORDS_ABBR = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06", "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}
    MONTHS_RUSSIAN = {"января": "01", "февраля": "02", "марта": "03", "апреля": "04","мая": "05", "июня": "06", "июля": "07", "августа": "08","сентября": "09", "октября": "10", "ноября": "11", "декабря": "12"}
    DAYS_TO_REPLACE = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

def clean_date():
    website = "kpru"
    df = pd.read_csv("../data/palestine/alquds_original.csv")
    
    pattern_en = r"\b(" + "|".join(Months.MONTHS_WORDS1.keys()) + r")\b"
    pattern_ru = r"\b(" + "|".join(Months.MONTHS_RUSSIAN.keys()) + r")\b"
    pattern_days = r"\b(" + "|".join(Months.DAYS_TO_REPLACE) + r")\b"
    pattern_abbr = r"\b(" + "|".join(Months.MONTHS_WORDS_ABBR.keys()) + r")\b"


    if website != "liganet":
        if website not in ["kpru", "alquds"]:
            df["date"] = (df["date"].str.replace(r'(?<=\w) (?=\w)', '', regex=True))
        df["date"] = (df["date"]
            .str.replace(pattern_days, "", regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_en, lambda m: Months.MONTHS_WORDS1[m.group(0).title()], regex=True, flags=re.IGNORECASE)
            .str.replace(pattern_ru, lambda m: Months.MONTHS_RUSSIAN[m.group(0).lower()], regex=True, flags=re.IGNORECASE)
            .str.replace(r"[,/]", " ", regex=True)
            .str.replace(r"\s+\d{1,2}\s*:\s*\d{2}.*$", "", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(pattern_abbr, lambda m: Months.MONTHS_WORDS_ABBR[m.group(0).title()], regex=True, flags=re.IGNORECASE)
            .str.strip())
        if website == "kpru":
            df["date"] = (df["date"].str.replace(" ", "-"))

        if website != "kpru":
            df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True).dt.strftime("%d-%m-%Y")
    print(df["date"].dtype)
    print(df["date"])

clean_date()


object
0     20-01-2026
1     20-01-2026
2     20-01-2026
3     20-01-2026
4     20-01-2026
         ...    
95    24-10-2025
96    23-10-2025
97    23-10-2025
98    22-10-2025
99    21-10-2025
Name: date, Length: 100, dtype: object


In [68]:
import matplotlib.pyplot as plt
website = "liganet"
df = pd.read_csv(f"../data/ukraine/{website}_embedded.csv")

filtered_df = df[(df["filter1"] > 0.4) | (df["filter2"] > 0.4) | (df["filter3"] > 0.4)].copy()
filtered_df.to_csv(f"../data/ukraine/{website}_filtered.csv")
print((filtered_df["title"].count() / df["title"].count())*100)

44.565217391304344


In [10]:
from transformers import pipeline

EMOTION_CLASSIFIER = pipeline(task = "text-classification", model = "j-hartmann/emotion-english-distilroberta-base", return_all_scores = True)
EMOTION_CLASSIFIER("But there is no doubt that the soldiers themselves will delay this moment to the last moment, knowing perfectly well that the weapons in Ukraine are fucked up and larger, and for any shot fired at the population, they will get at least five shots in their direction. It is not worth overestimating this phenomenon, considering it to be to some extent an expression of some ""prorussicity"" of Ukrainians. Until then, like ""before Kiev,"" and at least there is no ""pro-Russianity."" This is just an instinct for the self-preservation of civilians, albeit at a collective, higher level than that of singles. It is precisely for the system struggle against the regime of the Zelen and bandier power of the current ""cuffing"" of the population of the TCDs and the police that is clearly not enough.")

Device set to use cpu
c:\Users\sangi\Documents\GitHub\master-thesis\.venv\Lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


[[{'label': 'anger', 'score': 0.13161607086658478},
  {'label': 'disgust', 'score': 0.1095079556107521},
  {'label': 'fear', 'score': 0.006551758851855993},
  {'label': 'joy', 'score': 0.0023150774650275707},
  {'label': 'neutral', 'score': 0.7373749017715454},
  {'label': 'sadness', 'score': 0.00576796242967248},
  {'label': 'surprise', 'score': 0.0068662893027067184}]]

In [9]:
sentiment_model = pipeline("sentiment-analysis")
sentiment_model("But there is no doubt that the soldiers themselves will delay this moment to the last moment, knowing perfectly well that the weapons in Ukraine are fucked up and larger, and for any shot fired at the population, they will get at least five shots in their direction. It is not worth overestimating this phenomenon, considering it to be to some extent an expression of some ""prorussicity"" of Ukrainians. Until then, like ""before Kiev,"" and at least there is no ""pro-Russianity."" This is just an instinct for the self-preservation of civilians, albeit at a collective, higher level than that of singles. It is precisely for the system struggle against the regime of the Zelen and bandier power of the current ""cuffing"" of the population of the TCDs and the police that is clearly not enough.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\sangi\Documents\GitHub\master-thesis\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sangi\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' pac

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


[{'label': 'NEGATIVE', 'score': 0.9989326596260071}]

In [22]:
from transformers import pipeline
import numpy as np
import re

# Initialize models
sentiment_model = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=True
)

def analyze_phrase_entity_aware_regex(text, entities=None):
    """
    Analyze a single phrase:
    - Overall sentiment
    - Entity-targeted sentiment
    - Emotions
    Uses regex for sentence splitting, no NLTK required.
    """
    if entities is None:
        entities = []

    # --- 1️⃣ Overall sentiment ---
    s = sentiment_model(text)[0]
    overall = s['score'] if s['label'] == "POSITIVE" else -s['score']

    # --- 2️⃣ Entity-targeted sentiment ---
    def compute_entity_sentiment(text, entity):
        # Simple sentence splitting by ., !, ?
        sentences = re.split(r'[.!?]', text)
        relevant = [s for s in sentences if entity.lower() in s.lower()]
        if not relevant:
            return np.nan
        relevant_text = " ".join(relevant)
        s_ent = sentiment_model(relevant_text)[0]
        return s_ent['score'] if s_ent['label'] == "POSITIVE" else -s_ent['score']

    entity_sent = {entity: round(compute_entity_sentiment(text, entity), 3) for entity in entities}

    # --- 3️⃣ Emotions ---
    emotions = emotion_model(text)[0]
    em_dict = {f"emotion_{e['label'].lower()}": round(e['score'],3) for e in emotions}

    # Combine results
    result = {"sentiment_overall": round(overall,3)}
    result.update({f"sentiment_{entity.lower()}": score for entity, score in entity_sent.items()})
    result.update(em_dict)

    return result


phrase = """
The Russians do not need to recruit and buy Ukrainian officials – instead, they use reflexive management to make sure that corrupt officials fall into a prepared trap. One crime leads to another, bigger one. Those who say that Russian agents have been openly working in the government, parliament, and the Presidential Office for years are also right. People who are directly dependent on Moscow make key decisions. Much has been written publicly about their Russian connections, and they are still there. Now let's move to an even broader context. Those who say that the political crisis will weaken Ukraine tremendously are also right. We have a huge scandal of corruption and high treason in the immediate environment of the President of Ukraine.
"""

entities = ["Russia", "Ukraine"]

result = analyze_phrase_entity_aware_regex(phrase, entities=entities)
print(result)


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

c:\Users\sangi\Documents\GitHub\master-thesis\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sangi\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not insta

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cpu
Device set to use cpu


{'sentiment_overall': -0.753, 'sentiment_russia': -0.603, 'sentiment_ukraine': -0.879, 'emotion_anger': 0.163, 'emotion_disgust': 0.525, 'emotion_fear': 0.017, 'emotion_joy': 0.002, 'emotion_neutral': 0.278, 'emotion_sadness': 0.014, 'emotion_surprise': 0.002}
